<a href="https://colab.research.google.com/github/Saptaparnineogi/Customer-Churn-Prediction/blob/main/Telcom_Customer_Churn_Prediction_Model_building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files
import missingno as msno
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder

## Data Loading

In [32]:
uploadedfiles = files.upload()

Saving Churn.csv to Churn (2).csv


In [44]:
churn_df = pd.read_csv("Churn.csv")
churn_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data Processing

In [45]:
# Dropping customer id as it has no use
churn_df.drop("customerID", axis="columns", inplace=True)
# Deleting 11 rows with missing values for Total Charges
churn_df['TotalCharges'] = pd.to_numeric(churn_df.TotalCharges, errors='coerce')
churn_df = churn_df.dropna()
print(churn_df.shape)

(7032, 20)


In [46]:
def object_to_int(dataframe_series):
    if dataframe_series.dtype=='object':
        dataframe_series = LabelEncoder().fit_transform(dataframe_series)
    return dataframe_series

churn_df = churn_df.apply(lambda x: object_to_int(x))
churn_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


### Correlation among features

In [47]:
from statsmodels.stats.outliers_influence  import variance_inflation_factor
X = churn_df # your feature matrix
vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)

             feature        VIF
0             gender   1.954856
1      SeniorCitizen   1.373354
2            Partner   2.819317
3         Dependents   1.957986
4             tenure  15.086501
5       PhoneService  15.198554
6      MultipleLines   2.761146
7    InternetService   4.379748
8     OnlineSecurity   2.257841
9       OnlineBackup   2.459222
10  DeviceProtection   2.631343
11       TechSupport   2.392227
12       StreamingTV   3.237959
13   StreamingMovies   3.265624
14          Contract   4.210618
15  PaperlessBilling   2.891712
16     PaymentMethod   3.109158
17    MonthlyCharges  22.374797
18      TotalCharges  14.366392
19             Churn   1.861969


In [48]:
corr = churn_df[["PhoneService","TotalCharges", "tenure", "MonthlyCharges"]].corr()
fig = px.imshow( corr, text_auto=True, color_continuous_scale="RdBu_r", title="Correlation Matrix" )
fig.show()

#### Observation:

From the correlation matrix we see Tenure and Total charges are highly correlated, monthly charges and totda charges are also positively correlated.

Whereas for tree based methods it will not have an impact but it can impact adversely if we want to use some other algoritm for classification.

So to reduce the feature size and effect of multicolinearity (where it is required) I will remove Total Charges from the feature set.

In [49]:
churn_df.drop("TotalCharges", axis="columns", inplace=True)